# scRNA-seq analysis — MHC II expression in LUAD epithelial cells

**Goal:** Characterize MHC II expression in malignant vs normal adjacent epithelial
cells from LUAD patients, and compare expression across disease contexts including
primary tumor vs metastasis, early vs late stage, and AT2 vs Club cell identity.

**Input:** Salcher et al. 2022 pan-cancer scRNA-seq atlas (`local.h5ad`), subset to
lung adenocarcinoma epithelial lineage cells. Patient-level MHC II classifications
from IHC are used in figure 2e.

**Output:** Publication figures (main + supplemental) for figure 2 and associated
supplemental panels. Processed AnnData object with Harmony-corrected UMAP saved
to `outputs/processed/` for reuse across sessions.

## Setup

In [ ]:
# standard libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# single-cell analysis
import anndata as ad
import scanpy as sc

# clustering and statistics
from sklearn.cluster import AgglomerativeClustering
from scipy.stats import zscore, wilcoxon, mannwhitneyu
from scipy.cluster.hierarchy import dendrogram, linkage
import scipy.sparse as sp

# utilities
from tqdm import tqdm
from pathlib import Path
import yaml

# ceiba modules
from ceiba.plot_utils import sig_label

# figure settings
sns.set(font_scale=1.8)
sns.set_style('ticks')
plt.rcParams['pdf.fonttype'] = 42  # embed fonts in PDF output — required by most journals
plt.rcParams['ps.fonttype'] = 42
plt.rcParams.update({'axes.titlesize': 20})

# paper-wide color palettes — consistent across all analyses
# MHC II pos = orange (#FF8811), MHC II neg = purple (#462255)
cmap_low_high = ['#462255FF', '#FF8811FF', '#9DD9D2FF', '#046E8FFF', '#D44D5CFF']  # neg first
cmap_high_low = ['#FF8811FF', '#462255FF', '#9DD9D2FF', '#046E8FFF', '#D44D5CFF']  # pos first
cmap_umap     = ['#9DD9D2FF', '#462255FF', '#FF8811FF', '#046E8FFF', '#D44D5CFF']  # used in UMAPs

## Data loading

In [ ]:
import yaml
from pathlib import Path

# load paths configuration
with open('/home/gh8sj/projects/mhc2-luad-paper/data/paths_config.yml') as f:
    cfg = yaml.safe_load(f)

repo_root = Path(cfg['repo_root'])

# data paths
atlas_path     = cfg['datasets']['salcher_atlas']['raw']
processed_path = repo_root / cfg['outputs']['processed'] / 'luad_epithelial_harmony.h5ad'

# output paths — resolved against repo root
fig_dir   = repo_root / cfg['outputs']['figures']
table_dir = repo_root / cfg['outputs']['tables']

# create output directories if they don't exist
fig_out   = fig_dir / 'figure2'
table_out = table_dir / 'figure2'
fig_out.mkdir(parents=True, exist_ok=True)
table_out.mkdir(parents=True, exist_ok=True)

# load data — uses pre-processed Harmony object if available
# full atlas load is only required on first run or if reprocessing from scratch
if processed_path.exists():
    epithelial = sc.read_h5ad(processed_path)
    print(f'Loaded pre-processed epithelial object: {epithelial.n_obs:,} cells')
else:
    adata      = sc.read_h5ad(atlas_path)
    print(f'Loaded full atlas: {adata.n_obs:,} cells')
    # create epithelial subset so downstream cells always have epithelial defined
    # Harmony reintegration will overwrite this with the corrected embedding
    epithelial = adata[
        (adata.obs['disease'] == 'lung adenocarcinoma') &
        (adata.obs['ann_coarse'] == 'Epithelial cell')
    ].copy()
    print(f'Epithelial subset created: {epithelial.n_obs:,} cells')
    print('Run the Harmony reintegration section to compute embedding and save processed object')

In [ ]:
# add cancer_noncancer label if not already present
# this column is not saved with the processed object
if 'cancer_noncancer' not in epithelial.obs.columns:
    epithelial.obs['cancer_noncancer'] = np.where(
        epithelial.obs['ann_fine'] == 'Cancer cells',
        'Cancer cells',
        'Epithelial cells',
    )
    print('Added cancer_noncancer label')

# add numeric sample ID for batch visualization if not present
if 'sample_id' not in epithelial.obs.columns:
    epithelial.obs['sample_id'] = (
        epithelial.obs['sample']
        .map({s: i for i, s in enumerate(epithelial.obs['sample'].unique())})
        .astype(float)
    )
    print('Added sample_id')

## Gene lists

In [ ]:
# core MHC II antigen presentation genes
mhc2_genes = ['HLA-DRA', 'HLA-DRB1', 'HLA-DPA1', 'HLA-DPB1',
               'HLA-DQA1', 'HLA-DQB1', 'CD74', 'CIITA']

# extended set — adds accessory/co-stimulatory molecules and S100P
# S100P is included here as a comparator marker, not an MHC II gene
extended_genes = mhc2_genes + ['CD80', 'CD86', 'ICAM1', 'HLA-DMA', 'HLA-DMB', 'S100P']

def resolve_ensembl(adata, gene_list):
    """
    Map gene symbols to Ensembl IDs via var['feature_name'].

    Parameters
    ----------
    adata : AnnData
    gene_list : list of str
        Gene symbols to resolve.

    Returns
    -------
    dict : {gene_symbol: [ensembl_id, ...]}
    """
    out = {}
    for gene in gene_list:
        hits = list(adata.var.index[adata.var['feature_name'] == gene])
        if not hits:
            print(f'Warning: {gene} not found in var["feature_name"]')
        out[gene] = hits
    return out

    
# resolve gene symbols to Ensembl IDs — requires epithelial to be loaded
mhc2_dict     = resolve_ensembl(epithelial, mhc2_genes)
extended_dict = resolve_ensembl(epithelial, extended_genes)

# flat Ensembl ID lists for expression indexing
ens_mhc2     = [item for sublist in mhc2_dict.values()    for item in sublist]
ens_extended = [item for sublist in extended_dict.values() for item in sublist]

print(f'MHC II genes resolved:   {len(ens_mhc2)} Ensembl IDs')
print(f'Extended genes resolved: {len(ens_extended)} Ensembl IDs')

## Helper functions

Plotting functions used across figure 2 panels. `plot_genes_paired_luad` and
`plot_genes_paired_luad_percent_detected` operate on the full atlas object and
perform subsetting internally. `plot_scrna_group_comparison` operates on
pre-filtered AnnData subsets and is reused for figures 2c, 2d, and supplemental
comparisons. All three are pending migration to `ceiba.plot_utils`.

In [ ]:
# -------------------------------------------------------------------------
# Helper functions — paired expression plots for scRNA-seq data
# plot_genes_paired_luad and plot_genes_paired_luad_percent_detected
# are defined here pending migration to ceiba.plot_utils
# plot_scrna_group_comparison is defined here as it will be used
# for figures 2c, 2d, and supplemental comparisons
# -------------------------------------------------------------------------

def plot_genes_paired_luad(
    adata,
    genes,
    palette=('tab:grey', 'tab:red'),
    celltype='Epithelial cell',
    figsize_per_gene=(6, 5),
    nrows=1,
    test_mode='nonparametric',
    return_stats=False,
    title='',
    save_path=None,
):
    """
    Plot paired tumor vs normal adjacent expression for a list of genes.

    Supports any cell type in ann_coarse. For epithelial cells, tumor cells
    are restricted to ann_fine == 'Cancer cells' to exclude non-malignant
    epithelial cells in the tumor_primary compartment. For all other cell
    types, all cells of that type in tumor_primary are included.

    Only donors with both origins represented are included (paired design).
    Statistical test is selected based on test_mode — nonparametric (Wilcoxon)
    is recommended for consistency across genes.

    Parameters
    ----------
    adata : AnnData
        Atlas object. Subsetting to celltype and LUAD is performed internally.
    genes : list of str
        Gene symbols to plot (matched via var['feature_name']).
    palette : tuple
        Colors for (normal, tumor).
    celltype : str
        ann_coarse label to subset to (e.g. 'Epithelial cell',
        'Macrophage/Monocyte').
    figsize_per_gene : tuple
        (width, height) per panel.
    nrows : int
        Number of rows in the figure grid.
    test_mode : str
        'nonparametric' — Wilcoxon signed-rank (default, recommended).
        'parametric'    — paired t-test.
        'auto'          — Shapiro-Wilk normality test per gene, then select.
    return_stats : bool
        If True, return a DataFrame of per-gene statistics.
    title : str
        Optional figure-level title.
    save_path : Path or str
        If provided, save figure to this path.
    """
    from scipy.stats import shapiro, wilcoxon, ttest_rel

    # subset to LUAD and selected cell type
    sub = adata[
        (adata.obs['ann_coarse'] == celltype) &
        (adata.obs['disease'].astype(str).str.lower().str.replace('_', ' ')
         == 'lung adenocarcinoma')
    ].copy()
    sub.obs['origin']   = sub.obs['origin'].astype(str).str.strip().str.lower()
    sub.obs['ann_fine'] = sub.obs['ann_fine'].astype(str).str.strip().str.lower()

    normal_cells = sub[sub.obs['origin'] == 'normal_adjacent'].copy()

    # for epithelial cells restrict tumor subset to malignant cells only
    # for all other cell types include all cells of that type in tumor_primary
    if 'epithelial' in celltype.lower():
        tumor_cells = sub[
            (sub.obs['origin'] == 'tumor_primary') &
            (sub.obs['ann_fine'] == 'cancer cells')
        ].copy()
    else:
        tumor_cells = sub[sub.obs['origin'] == 'tumor_primary'].copy()

    sub = ad.concat([normal_cells, tumor_cells], axis=0, merge='same')

    ncols     = int(np.ceil(len(genes) / nrows))
    fig, axes = plt.subplots(
        nrows=nrows, ncols=ncols,
        figsize=(figsize_per_gene[0] * ncols, figsize_per_gene[1] * nrows),
        sharey=False,
    )
    axes = np.ravel(axes)

    stats_list = []

    for ax, gene in zip(axes, genes):
        try:
            gene_id = sub.var.loc[sub.var['feature_name'] == gene].index[0]
        except IndexError:
            print(f'{gene} not found in var["feature_name"] — skipping')
            ax.axis('off')
            continue

        x = sub[:, gene_id].X
        x = x.toarray().ravel() if hasattr(x, 'toarray') else np.asarray(x).ravel()

        df = (
            pd.DataFrame({
                'donor_id': sub.obs['donor_id'].astype(str).values,
                'origin':   sub.obs['origin'].values,
                gene:       x,
            })
            .groupby(['donor_id', 'origin'], observed=True)[gene]
            .mean()
            .reset_index()
        )

        donor_counts  = df.groupby('donor_id')['origin'].nunique()
        paired_donors = donor_counts[donor_counts == 2].index
        df = df[df['donor_id'].isin(paired_donors)]
        if df.empty:
            print(f'No paired donors for {gene} — skipping')
            ax.axis('off')
            continue

        order = ['normal_adjacent', 'tumor_primary']

        sns.violinplot(
            data=df, x='origin', y=gene, hue='origin',
            order=order, palette=palette,
            inner=None, linewidth=1.2, cut=0, fill=False,
            ax=ax, legend=False,
        )
        sns.stripplot(
            data=df, x='origin', y=gene, hue='origin',
            order=order, palette=palette,
            dodge=False, size=6, alpha=0.7,
            ax=ax, legend=False,
        )

        normal_vals, tumor_vals = [], []
        for did, vals in df.groupby('donor_id'):
            norm_val  = vals.loc[vals['origin'] == 'normal_adjacent', gene].values[0]
            tumor_val = vals.loc[vals['origin'] == 'tumor_primary',   gene].values[0]
            ax.plot([0, 1], [norm_val, tumor_val], color='gray', alpha=0.4, linewidth=0.8)
            normal_vals.append(norm_val)
            tumor_vals.append(tumor_val)

        diff   = np.array(tumor_vals) - np.array(normal_vals)
        p_norm = np.nan

        if test_mode == 'auto':
            if len(diff) >= 3:
                _, p_norm = shapiro(diff)
            if np.isnan(p_norm) or p_norm <= 0.05:
                test_name, (stat, p) = 'Wilcoxon', wilcoxon(tumor_vals, normal_vals)
            else:
                test_name, (stat, p) = 'Paired t-test', ttest_rel(tumor_vals, normal_vals)
        elif test_mode == 'parametric':
            test_name, (stat, p) = 'Paired t-test', ttest_rel(tumor_vals, normal_vals)
        elif test_mode == 'nonparametric':
            test_name, (stat, p) = 'Wilcoxon', wilcoxon(tumor_vals, normal_vals)
        else:
            raise ValueError("test_mode must be 'auto', 'parametric', or 'nonparametric'")

        star = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
        ymax  = df[gene].max()
        yoff  = (df[gene].max() - df[gene].min()) * 0.15
        ax.text(0.5, ymax + yoff, star, ha='center', va='bottom',
                fontsize=28, fontweight='bold')
        ax.set_ylim(top=ymax + yoff * 2)
        ax.set_title(gene)
        ax.set_xticks([0, 1])
        ax.set_xticklabels(['Normal\nAdjacent', 'Primary\nTumor'])
        ax.set_xlabel('')
        ax.set_ylabel('Mean expression' if ax == axes[0] else '')
        ax.spines[['top', 'right']].set_visible(False)

        stats_list.append({
            'Gene':         gene,
            'n_pairs':      len(diff),
            'Normality_p':  p_norm,
            'Test':         test_name,
            'Stat':         stat,
            'p_value':      p,
        })

    for ax in axes[len(genes):]:
        ax.set_visible(False)

    if title:
        fig.suptitle(title, fontsize=24, fontweight='bold', y=1.03)

    plt.tight_layout(rect=[0, 0, 1, 0.97])

    if save_path:
        fig.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f'Saved → {save_path}')
    plt.show()

    if return_stats:
        return pd.DataFrame(stats_list)


def plot_genes_paired_luad_percent_detected(
    adata,
    genes,
    palette=('tab:grey', 'tab:red'),
    celltype='Epithelial cell',
    detection_thresh=0.0,
    figsize_per_gene=(6, 5),
    return_stats=False,
    save_path=None,
):
    """
    Plot the percent of cells per sample with detected expression (> detection_thresh)
    for each gene, comparing normal adjacent vs primary tumor in paired LUAD donors.

    Complements plot_genes_paired_luad — percent detection is robust to zero-inflation
    and captures the binary presence/absence signal independently of mean expression level.

    Parameters
    ----------
    adata : AnnData
        Full atlas object. Subsetting is performed internally.
    genes : list of str
        Gene symbols to plot (matched via var['feature_name']).
    palette : tuple
        Colors for (normal, tumor).
    celltype : str
        ann_coarse label to subset to.
    detection_thresh : float
        Cells with expression > this value are counted as detected (default 0.0).
    figsize_per_gene : tuple
        (width, height) per panel.
    return_stats : bool
        If True, return a DataFrame of per-gene statistics.
    save_path : Path or str
        If provided, save figure to this path.
    """
    from scipy.stats import wilcoxon

    n_genes = len(genes)
    fig, axes = plt.subplots(
        nrows=1, ncols=n_genes,
        figsize=(figsize_per_gene[0] * n_genes, figsize_per_gene[1]),
        sharey=False,
    )
    axes = np.atleast_1d(axes).flatten()

    stats_list = []

    for ax, gene in zip(axes, genes):
        epi = adata[adata.obs['ann_coarse'] == celltype].copy()
        epi = epi[
            epi.obs['disease'].astype(str).str.lower().str.replace('_', ' ')
            == 'lung adenocarcinoma'
        ].copy()
        epi.obs['origin'] = epi.obs['origin'].astype(str).str.strip().str.lower()

        normal_epi = epi[epi.obs['origin'] == 'normal_adjacent'].copy()
        tumor_epi  = epi[
            (epi.obs['origin'] == 'tumor_primary') &
            (epi.obs['ann_fine'].astype(str).str.lower() == 'cancer cells')
        ].copy()
        epi = ad.concat([normal_epi, tumor_epi], axis=0, merge='same')

        try:
            gene_id = epi.var.loc[epi.var['feature_name'] == gene].index[0]
        except IndexError:
            print(f'{gene} not found in var["feature_name"] — skipping')
            ax.axis('off')
            continue

        x = epi[:, gene_id].X
        x = x.toarray().ravel() if hasattr(x, 'toarray') else np.asarray(x).ravel()

        # fraction of cells per sample with detected expression
        df = (
            pd.DataFrame({
                'donor_id': epi.obs['donor_id'].astype(str).values,
                'sample':   epi.obs['sample'].astype(str).values,
                'origin':   epi.obs['origin'].values,
                'detected': (x > detection_thresh).astype(int),
            })
            .groupby(['donor_id', 'sample', 'origin'], observed=True)['detected']
            .mean()
            .mul(100.0)
            .reset_index()
            .rename(columns={'detected': 'pct_detected'})
        )

        donor_counts  = df.groupby('donor_id')['origin'].nunique()
        paired_donors = donor_counts[donor_counts == 2].index
        df = df[df['donor_id'].isin(paired_donors)]
        if df.empty:
            ax.axis('off')
            continue

        order = ['normal_adjacent', 'tumor_primary']

        sns.violinplot(
            data=df, x='origin', y='pct_detected', hue='origin',
            order=order, palette=palette,
            inner=None, linewidth=1.2, cut=0, fill=False,
            ax=ax, legend=False,
        )
        sns.stripplot(
            data=df, x='origin', y='pct_detected', hue='origin',
            order=order, palette=palette,
            dodge=False, size=8, alpha=0.7,
            ax=ax, legend=False,
        )

        tumor_vals, normal_vals = [], []
        for did, vals in df.groupby('donor_id'):
            if set(order).issubset(set(vals['origin'].values)):
                norm_val  = vals.loc[vals['origin'] == 'normal_adjacent', 'pct_detected'].values[0]
                tumor_val = vals.loc[vals['origin'] == 'tumor_primary',   'pct_detected'].values[0]
                ax.plot([0, 1], [norm_val, tumor_val], color='gray', alpha=0.4, linewidth=0.8)
                normal_vals.append(norm_val)
                tumor_vals.append(tumor_val)

        if tumor_vals and normal_vals:
            stat, p = wilcoxon(tumor_vals, normal_vals, alternative='two-sided')
            star = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
        else:
            p, star = np.nan, 'ns'

        ymax = df['pct_detected'].max()
        ax.text(0.5, ymax * 0.85, star, ha='center', va='bottom', fontsize=32)

        ax.set_title(gene)
        ax.set_xticks(range(len(order)))
        ax.set_xticklabels(['Normal\nAdjacent', 'Primary\nTumor'])
        ax.set_xlabel('')
        ax.set_ylabel('% cells detected' if ax == axes[0] else '')
        ax.spines[['top', 'right']].set_visible(False)

        stats_list.append({'Gene': gene, 'n_pairs': len(tumor_vals), 'Wilcoxon_p': p})

    for ax in axes[len(genes):]:
        ax.set_visible(False)

    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f'Saved → {save_path}')
    plt.show()

    if return_stats:
        return pd.DataFrame(stats_list)


def plot_scrna_group_comparison(
    adata,
    genes,
    group_col,
    order,
    palette,
    xtick_labels,
    gene_label_col='feature_name',
    figsize_per_gene=(5, 5),
    nrows=1,
    point_size=5,
    save_path=None,
    dpi=300,
):
    """
    Compare per-sample mean expression of a gene set between two or more groups.

    Aggregates single-cell expression to sample-level means, then plots a
    violin + strip panel per gene with Mann-Whitney U significance annotation.
    Designed for comparing MHC II expression across disease contexts
    (e.g. primary vs metastasis, early vs late stage).

    Parameters
    ----------
    adata : AnnData
        Pre-filtered object containing only the cells to include.
        Group labels must already be present in adata.obs[group_col].
    genes : list of str
        Ensembl IDs to plot (matched to adata.var_names).
    group_col : str
        obs column defining comparison groups (e.g. 'origin', 'stage_group').
    order : list of str
        Group values in display order.
    palette : dict
        Mapping of group value to color.
    xtick_labels : list of str
        Display labels for x-axis ticks, corresponding to order.
    gene_label_col : str
        var column used to label each panel (default 'feature_name').
    figsize_per_gene : tuple
        (width, height) per panel.
    nrows : int
        Number of rows in the figure grid.
    point_size : int
        Marker size for strip points.
    save_path : Path or str or None
        If provided, save figure to this path.
    dpi : int
        Resolution for raster outputs.
    """
    from scipy.stats import mannwhitneyu

    # aggregate to sample-level mean expression
    expr_df            = adata.to_df()[genes].copy()
    expr_df['sample']  = adata.obs['sample'].values
    expr_df[group_col] = adata.obs[group_col].values
    sample_avg         = expr_df.groupby('sample', observed=True)[genes].mean()

    # map each sample to its group (mode, in case of mixed obs)
    sample_group = (
        adata.obs
        .groupby('sample', observed=True)[group_col]
        .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan)
        .dropna()
    )
    sample_avg[group_col] = sample_group
    sample_avg            = sample_avg.dropna(subset=[group_col])

    ncols     = int(np.ceil(len(genes) / nrows))
    fig, axes = plt.subplots(
        nrows, ncols,
        figsize=(figsize_per_gene[0] * ncols, figsize_per_gene[1] * nrows),
    )
    axes = np.atleast_1d(axes).flatten()

    for ax, gene in zip(axes, genes):
        plot_df = sample_avg[[gene, group_col]].dropna()

        if not set(order).issubset(plot_df[group_col].unique()):
            ax.set_title('insufficient data')
            ax.axis('off')
            continue

        sns.violinplot(
            data=plot_df, x=group_col, y=gene, hue=group_col,
            order=order, palette=palette,
            inner=None, fill=False, linewidth=1.2, cut=0,
            ax=ax, legend=False,
        )
        sns.stripplot(
            data=plot_df, x=group_col, y=gene, hue=group_col,
            order=order, palette=palette,
            edgecolor='k', linewidth=1, size=point_size,
            ax=ax, legend=False,
        )

        # resolve display label from var
        feature_label = (
            adata.var.loc[gene, gene_label_col]
            if gene in adata.var.index and gene_label_col in adata.var.columns
            else gene
        )
        ax.set_title(feature_label)
        ax.set_xlabel('')
        ax.set_ylabel('')
        ax.set_xticks(range(len(order)))
        ax.set_xticklabels(xtick_labels)
        ax.spines[['top', 'right']].set_visible(False)

        # Mann-Whitney U — two-sided, sample level
        g1 = plot_df.loc[plot_df[group_col] == order[0], gene]
        g2 = plot_df.loc[plot_df[group_col] == order[1], gene]
        _, pval = mannwhitneyu(g1, g2, alternative='two-sided')
        star = '***' if pval < 0.001 else '**' if pval < 0.01 else '*' if pval < 0.05 else 'n.s.'
        ax.text(0.5, 0.85, star, ha='center', va='bottom',
                fontsize=32, transform=ax.transAxes)

    for ax in axes[len(genes):]:
        ax.set_visible(False)

    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=dpi, bbox_inches='tight')
        print(f'Saved → {save_path}')
    plt.show()

def plot_genes_pct_expressing_luad(
    adata,
    genes,
    palette=('tab:grey', 'tab:red'),
    celltype='Epithelial cell',
    figsize_per_gene=(6, 5),
    nrows=1,
    test_mode='nonparametric',
    return_stats=False,
    title='',
    save_path=None,
):
    """
    Plot the percent of cells per donor expressing each gene (expression > 0),
    comparing normal adjacent vs primary tumor in paired LUAD donors.

    Complements plot_genes_paired_luad — percent detection is robust to
    zero-inflation and captures presence/absence signal independently of
    mean expression magnitude.

    Supports any cell type in ann_coarse. For epithelial cells, tumor cells
    are restricted to ann_fine == 'Cancer cells'. For all other cell types,
    all cells of that type in tumor_primary are included.

    Parameters
    ----------
    adata : AnnData
        Atlas object. Subsetting to celltype and LUAD is performed internally.
    genes : list of str
        Gene symbols to plot (matched via var['feature_name']).
    palette : tuple
        Colors for (normal, tumor).
    celltype : str
        ann_coarse label to subset to.
    figsize_per_gene : tuple
        (width, height) per panel.
    nrows : int
        Number of rows in the figure grid.
    test_mode : str
        'nonparametric' — Wilcoxon signed-rank (default, recommended).
        'parametric'    — paired t-test.
        'auto'          — Shapiro-Wilk normality test per gene, then select.
    return_stats : bool
        If True, return a DataFrame of per-gene statistics.
    title : str
        Optional figure-level title.
    save_path : Path or str
        If provided, save figure to this path.
    """
    from scipy.stats import shapiro, wilcoxon, ttest_rel

    sub = adata[
        (adata.obs['ann_coarse'] == celltype) &
        (adata.obs['disease'].astype(str).str.lower().str.replace('_', ' ')
         == 'lung adenocarcinoma')
    ].copy()
    sub.obs['origin']   = sub.obs['origin'].astype(str).str.strip().str.lower()
    sub.obs['ann_fine'] = sub.obs['ann_fine'].astype(str).str.strip().str.lower()

    normal_cells = sub[sub.obs['origin'] == 'normal_adjacent'].copy()

    if 'epithelial' in celltype.lower():
        tumor_cells = sub[
            (sub.obs['origin'] == 'tumor_primary') &
            (sub.obs['ann_fine'] == 'cancer cells')
        ].copy()
    else:
        tumor_cells = sub[sub.obs['origin'] == 'tumor_primary'].copy()

    sub = ad.concat([normal_cells, tumor_cells], axis=0, merge='same')

    ncols     = int(np.ceil(len(genes) / nrows))
    fig, axes = plt.subplots(
        nrows=nrows, ncols=ncols,
        figsize=(figsize_per_gene[0] * ncols, figsize_per_gene[1] * nrows),
        sharey=False,
    )
    axes = np.ravel(axes)

    stats_list = []

    for ax, gene in zip(axes, genes):
        try:
            gene_id = sub.var.loc[sub.var['feature_name'] == gene].index[0]
        except IndexError:
            print(f'{gene} not found in var["feature_name"] — skipping')
            ax.axis('off')
            continue

        x = sub[:, gene_id].X
        x = x.toarray().ravel() if hasattr(x, 'toarray') else np.asarray(x).ravel()

        # fraction of cells per donor with any detected expression
        df = (
            pd.DataFrame({
                'donor_id': sub.obs['donor_id'].astype(str).values,
                'origin':   sub.obs['origin'].values,
                'detected': (x > 0).astype(float),
            })
            .groupby(['donor_id', 'origin'], observed=True)['detected']
            .mean()
            .mul(100.0)
            .reset_index()
            .rename(columns={'detected': 'pct_detected'})
        )

        donor_counts  = df.groupby('donor_id')['origin'].nunique()
        paired_donors = donor_counts[donor_counts == 2].index
        df = df[df['donor_id'].isin(paired_donors)]
        if df.empty:
            print(f'No paired donors for {gene} — skipping')
            ax.axis('off')
            continue

        order = ['normal_adjacent', 'tumor_primary']

        sns.violinplot(
            data=df, x='origin', y='pct_detected', hue='origin',
            order=order, palette=palette,
            inner=None, linewidth=1.2, cut=0, fill=False,
            ax=ax, legend=False,
        )
        sns.stripplot(
            data=df, x='origin', y='pct_detected', hue='origin',
            order=order, palette=palette,
            dodge=False, size=6, alpha=0.7,
            ax=ax, legend=False,
        )

        normal_vals, tumor_vals = [], []
        for did, vals in df.groupby('donor_id'):
            norm_val  = vals.loc[vals['origin'] == 'normal_adjacent', 'pct_detected'].values[0]
            tumor_val = vals.loc[vals['origin'] == 'tumor_primary',   'pct_detected'].values[0]
            ax.plot([0, 1], [norm_val, tumor_val], color='gray', alpha=0.4, linewidth=0.8)
            normal_vals.append(norm_val)
            tumor_vals.append(tumor_val)

        diff   = np.array(tumor_vals) - np.array(normal_vals)
        p_norm = np.nan

        if test_mode == 'auto':
            if len(diff) >= 3:
                _, p_norm = shapiro(diff)
            if np.isnan(p_norm) or p_norm <= 0.05:
                test_name, (stat, p) = 'Wilcoxon', wilcoxon(tumor_vals, normal_vals)
            else:
                test_name, (stat, p) = 'Paired t-test', ttest_rel(tumor_vals, normal_vals)
        elif test_mode == 'parametric':
            test_name, (stat, p) = 'Paired t-test', ttest_rel(tumor_vals, normal_vals)
        elif test_mode == 'nonparametric':
            test_name, (stat, p) = 'Wilcoxon', wilcoxon(tumor_vals, normal_vals)
        else:
            raise ValueError("test_mode must be 'auto', 'parametric', or 'nonparametric'")

        star = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
        ymax  = df['pct_detected'].max()
        yoff  = (df['pct_detected'].max() - df['pct_detected'].min()) * 0.15
        ax.text(0.5, ymax + yoff, star, ha='center', va='bottom',
                fontsize=28, fontweight='bold')
        ax.set_ylim(top=ymax + yoff * 2)
        ax.set_title(gene)
        ax.set_xticks([0, 1])
        ax.set_xticklabels(['Normal\nAdjacent', 'Primary\nTumor'])
        ax.set_xlabel('')
        ax.set_ylabel('% cells expressing' if ax == axes[0] else '')
        ax.spines[['top', 'right']].set_visible(False)

        stats_list.append({
            'Gene':        gene,
            'n_pairs':     len(diff),
            'Normality_p': p_norm,
            'Test':        test_name,
            'Stat':        stat,
            'p_value':     p,
        })

    for ax in axes[len(genes):]:
        ax.set_visible(False)

    if title:
        fig.suptitle(title, fontsize=24, fontweight='bold', y=1.03)

    plt.tight_layout(rect=[0, 0, 1, 0.97])

    if save_path:
        fig.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f'Saved → {save_path}')
    plt.show()

    if return_stats:
        return pd.DataFrame(stats_list)

def plot_celltype_comparison_luad(
    adata,
    genes,
    tissue='tumor_primary',
    celltypes=('Epithelial cell', 'Macrophage/Monocyte'),
    palette=('#9DD9D2', '#D44D5C'),
    figsize_per_gene=(6, 5),
    nrows=1,
    test_mode='nonparametric',
    return_stats=False,
    title='',
    save_path=None,
):
    """
    Compare gene expression between two cell types within the same tissue
    region (e.g. epithelial vs macrophage in primary tumor or NAT).

    Donors are paired — only donors with both cell types represented in the
    specified tissue are included. Useful for showing that MHC II expression
    in malignant epithelial cells is comparable to professional APCs.

    Parameters
    ----------
    adata : AnnData
        Atlas object containing LUAD cells.
    genes : list of str
        Gene symbols to plot (matched via var['feature_name']).
    tissue : str
        Origin value to filter to ('tumor_primary' or 'normal_adjacent').
    celltypes : tuple of str
        Two ann_coarse labels to compare.
    palette : tuple or str
        Colors for each cell type, or a seaborn palette name.
    figsize_per_gene : tuple
        (width, height) per panel.
    nrows : int
        Number of rows in the figure grid.
    test_mode : str
        'nonparametric' — Wilcoxon signed-rank (default, recommended).
        'parametric'    — paired t-test.
        'auto'          — Shapiro-Wilk normality test per gene, then select.
    return_stats : bool
        If True, return a DataFrame of per-gene statistics.
    title : str
        Optional figure-level title.
    save_path : Path or str
        If provided, save figure to this path.
    """
    from scipy.stats import shapiro, wilcoxon, ttest_rel

    # subset to LUAD and specified tissue region
    sub = adata[
        (adata.obs['origin'].str.lower() == tissue.lower()) &
        (adata.obs['disease'].astype(str).str.lower().str.replace('_', ' ')
         == 'lung adenocarcinoma')
    ].copy()
    sub.obs['ann_coarse'] = sub.obs['ann_coarse'].astype(str).str.strip()

    ncols     = int(np.ceil(len(genes) / nrows))
    fig, axes = plt.subplots(
        nrows=nrows, ncols=ncols,
        figsize=(figsize_per_gene[0] * ncols, figsize_per_gene[1] * nrows),
        sharey=False,
    )
    axes = np.ravel(axes)

    stats_list = []

    for ax, gene in zip(axes, genes):
        try:
            gene_id = sub.var.loc[sub.var['feature_name'] == gene].index[0]
        except IndexError:
            print(f'{gene} not found in var["feature_name"] — skipping')
            ax.axis('off')
            continue

        x = sub[:, gene_id].X
        x = x.toarray().ravel() if hasattr(x, 'toarray') else np.asarray(x).ravel()

        df = (
            pd.DataFrame({
                'donor_id': sub.obs['donor_id'].astype(str).values,
                'celltype': sub.obs['ann_coarse'].values,
                gene:       x,
            })
            .loc[lambda d: d['celltype'].isin(celltypes)]
            .groupby(['donor_id', 'celltype'], observed=True)[gene]
            .mean()
            .reset_index()
        )

        # retain donors with both cell types represented
        donor_counts  = df.groupby('donor_id')['celltype'].nunique()
        paired_donors = donor_counts[donor_counts == 2].index
        df = df[df['donor_id'].isin(paired_donors)]
        if df.empty:
            print(f'No paired donors for {gene} — skipping')
            ax.axis('off')
            continue

        sns.violinplot(
            data=df, x='celltype', y=gene, hue='celltype',
            order=celltypes, palette=palette,
            inner=None, linewidth=1.2, cut=0, fill=False,
            ax=ax, legend=False,
        )
        sns.stripplot(
            data=df, x='celltype', y=gene, hue='celltype',
            order=celltypes, palette=palette,
            dodge=False, size=7, alpha=0.7,
            ax=ax, legend=False,
        )

        vals1, vals2 = [], []
        for did, vals in df.groupby('donor_id'):
            v1 = vals.loc[vals['celltype'] == celltypes[0], gene].values[0]
            v2 = vals.loc[vals['celltype'] == celltypes[1], gene].values[0]
            ax.plot([0, 1], [v1, v2], color='gray', alpha=0.4, linewidth=0.8)
            vals1.append(v1)
            vals2.append(v2)

        diff   = np.array(vals2) - np.array(vals1)
        p_norm = np.nan

        if test_mode == 'auto':
            if len(diff) >= 3:
                _, p_norm = shapiro(diff)
            if np.isnan(p_norm) or p_norm <= 0.05:
                test_name, (stat, p) = 'Wilcoxon', wilcoxon(vals2, vals1)
            else:
                test_name, (stat, p) = 'Paired t-test', ttest_rel(vals2, vals1)
        elif test_mode == 'parametric':
            test_name, (stat, p) = 'Paired t-test', ttest_rel(vals2, vals1)
        elif test_mode == 'nonparametric':
            test_name, (stat, p) = 'Wilcoxon', wilcoxon(vals2, vals1)
        else:
            raise ValueError("test_mode must be 'auto', 'parametric', or 'nonparametric'")

        star = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
        ymax  = df[gene].max()
        yoff  = (df[gene].max() - df[gene].min()) * 0.2
        ax.text(0.5, ymax + yoff, star, ha='center', va='bottom', fontsize=26)
        ax.set_ylim(top=ymax + yoff * 2)
        ax.set_title(f'{gene} ({tissue})')
        ax.set_xticks([0, 1])
        ax.set_xticklabels(list(celltypes))
        ax.set_xlabel('')
        ax.set_ylabel('Mean expression' if ax == axes[0] else '')
        ax.spines[['top', 'right']].set_visible(False)

        stats_list.append({
            'Gene':        gene,
            'tissue':      tissue,
            'n_pairs':     len(diff),
            'Test':        test_name,
            'Stat':        stat,
            'p_value':     p,
            'Normality_p': p_norm,
        })

    for ax in axes[len(genes):]:
        ax.set_visible(False)

    if title:
        fig.suptitle(title, fontsize=24, fontweight='bold', y=1.03)

    plt.tight_layout(rect=[0, 0, 1, 0.97])

    if save_path:
        fig.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f'Saved → {save_path}')
    plt.show()

    if return_stats:
        return pd.DataFrame(stats_list)

## Epithelial lineage subsetting and Harmony reintegration

The Salcher atlas UMAP embeds all cell types together. When subsetting to
epithelial lineage cells only, the original embedding collapses into a dense
cluster that obscures within-lineage structure. A new UMAP is computed on the
epithelial subset using Harmony batch correction across studies.

This section only needs to run once. The processed object is saved to
`outputs/processed/epithelial_harmony.h5ad` and loaded automatically on
subsequent sessions.

In [ ]:
# guard — skip if processed object was already loaded
if 'epithelial' not in dir():

    # subset to LUAD epithelial lineage cells
    adenodata  = adata[adata.obs['disease'] == 'lung adenocarcinoma'].copy()
    epithelial = adenodata[adenodata.obs['ann_coarse'] == 'Epithelial cell'].copy()

    # label cancer vs non-cancer epithelial cells
    epithelial.obs['cancer_noncancer'] = np.where(
        epithelial.obs['ann_fine'] == 'Cancer cells',
        'Cancer cells',
        'Epithelial cells',
    )

    print(f'Epithelial lineage cells: {epithelial.n_obs:,}')
    print(epithelial.obs['cancer_noncancer'].value_counts())

In [ ]:
if 'X_pca_harmony' not in epithelial.obsm:
    import time
    import harmonypy

    # HVG selection
    print('Step 1/5 — highly variable genes...')
    t0 = time.time()
    sc.pp.highly_variable_genes(epithelial, n_top_genes=2000, subset=False)
    print(f'  done ({time.time()-t0:.1f}s) — {epithelial.var.highly_variable.sum()} HVGs selected')

    # PCA
    print('Step 2/5 — PCA...')
    t0 = time.time()
    sc.tl.pca(epithelial, mask_var='highly_variable')
    print(f'  done ({time.time()-t0:.1f}s)')

    # Harmony — called directly due to anndata/harmonypy version incompatibility
    # in this version of harmonypy, Z_corr is already (n_cells, n_pcs) — no transpose needed
    print('Step 3/5 — Harmony batch correction (this is the slow step)...')
    t0 = time.time()
    ho = harmonypy.run_harmony(
        epithelial.obsm['X_pca'].astype('float64'),
        epithelial.obs,
        'study',
        verbose=True,
    )
    epithelial.obsm['X_pca_harmony'] = ho.Z_corr
    print(f'  done ({time.time()-t0:.1f}s)')
    print(f'  embedding shape: {epithelial.obsm["X_pca_harmony"].shape}')

    # neighborhood graph
    print('Step 4/5 — neighborhood graph...')
    t0 = time.time()
    sc.pp.neighbors(epithelial, use_rep='X_pca_harmony', n_neighbors=10, n_pcs=20)
    print(f'  done ({time.time()-t0:.1f}s)')

    # UMAP and Leiden clustering
    print('Step 5/5 — UMAP and Leiden clustering...')
    t0 = time.time()
    sc.tl.umap(epithelial)
    sc.tl.leiden(epithelial, resolution=0.5)
    print(f'  done ({time.time()-t0:.1f}s)')

    # numeric sample ID for batch visualization
    epithelial.obs['sample_id'] = (
        epithelial.obs['sample']
        .map({s: i for i, s in enumerate(epithelial.obs['sample'].unique())})
        .astype(float)
    )

    # save epithelial object — subsequent sessions load this directly
    out_path = repo_root / cfg['outputs']['processed'] / 'luad_epithelial_harmony.h5ad'
    out_path.parent.mkdir(parents=True, exist_ok=True)
    epithelial.write(out_path)
    print(f'Saved epithelial object → {out_path}')

    # save full LUAD object — all cell types, used for supplemental analyses
    # no Harmony needed here — used for expression comparisons, not UMAP
    luad = adata[
        adata.obs['disease'] == 'lung adenocarcinoma'
    ].copy()
    luad = luad[luad.obs['origin'].isin(['tumor_primary', 'normal_adjacent'])].copy()
    luad_path = repo_root / cfg['outputs']['processed'] / 'luad.h5ad'
    luad.write(luad_path)
    print(f'Saved full LUAD object → {luad_path}')

else:
    print('Harmony embedding already present — skipping reintegration')

## Analysis subsets

All downstream figure subsets are derived from `epithelial` here in one place,
so that UMAPs and expression comparisons always operate on the same cells.

In [ ]:
# paired LUAD donors — used for figure 2a (tumor vs NAT comparison)
luad = epithelial[epithelial.obs['origin'].isin(['tumor_primary', 'normal_adjacent'])].copy()

paired_donors = (
    luad.obs
    .groupby('donor_id')['origin']
    .nunique()
)
paired_donors = paired_donors[paired_donors == 2].index
luad_paired   = luad[luad.obs['donor_id'].isin(paired_donors)].copy()

print(f'Paired donors:   {len(paired_donors)}')
print(f'Cells retained:  {luad_paired.n_obs:,}')

In [ ]:
# cancer cells from primary tumor + metastasis — used for figure 2c
met_nonmet = epithelial[
    epithelial.obs['origin'].isin(['tumor_primary', 'tumor_metastasis']) &
    (epithelial.obs['ann_fine'] == 'Cancer cells')
].copy()

# primary tumor cancer cells only — used for figure 2d (stage comparison)
# stage mapped to early (I/II) vs late (III/IV)
nonmet = epithelial[
    (epithelial.obs['origin'] == 'tumor_primary') &
    (epithelial.obs['ann_fine'] == 'Cancer cells')
].copy()

nonmet.obs['stage_group'] = nonmet.obs['uicc_stage'].map(
    lambda x: 'Early' if x in ['I', 'II'] else 'Late'
)

print(f'met_nonmet:  {met_nonmet.n_obs:,} cells')
print(f'nonmet:      {nonmet.n_obs:,} cells')
print(f'Stage distribution:\n{nonmet.obs["stage_group"].value_counts()}')

## UMAPs

QC plots confirm Harmony integration quality before generating figure panels.
Figure UMAPs are saved to `outputs/figures/figure2/` and sit alongside the
corresponding expression comparison panels in the final figure layout.

In [ ]:
# QC — verify Harmony integration
# cells should mix across studies and samples with no strong batch clustering
sc.pl.umap(epithelial, color='sample_id', cmap='rainbow',
           title='Samples (batch check)', s=5, alpha=0.9)
sc.pl.umap(epithelial, color='study', title='Study (batch check)', s=5, alpha=0.5)

In [ ]:
# set scanpy figure output directory
sc.settings.figdir = str(fig_out)

In [ ]:
# figure 2 context UMAP — cancer vs non-cancer epithelial cells
sc.pl.umap(
    epithelial,
    color='cancer_noncancer',
    palette={'Cancer cells': 'tab:red', 'Epithelial cells': 'tab:grey'},
    title='Epithelial lineage',
    s=5, alpha=0.8,
    save='_figure2_umap_cancer_noncancer.pdf',
)


In [ ]:
# cancer cells from primary tumor + metastasis — used for figure 2c
met_nonmet = epithelial[
    epithelial.obs['origin'].isin(['tumor_primary', 'tumor_metastasis']) &
    (epithelial.obs['ann_fine'] == 'Cancer cells')
].copy()

# rename origin labels for display
met_nonmet.obs['origin'] = (
    met_nonmet.obs['origin']
    .astype('category')
    .cat.rename_categories({
        'tumor_primary':    'Primary Tumor',
        'tumor_metastasis': 'Metastasis',
    })
)

print(f'met_nonmet: {met_nonmet.n_obs:,} cells')
print(met_nonmet.obs['origin'].value_counts())


In [ ]:
# figure 2c context UMAP
sc.pl.umap(
    met_nonmet,
    color='origin',
    palette={'Primary Tumor': 'goldenrod', 'Metastasis': 'red'},
    title='Primary vs metastasis',
    s=3, alpha=0.7,
    show=False,
)

In [ ]:

# figure 2d context UMAP — early vs late stage primary tumor cancer cells
sc.pl.umap(
    nonmet,
    color='stage_group',
    palette = {'Early': '#f8a5c2', 'Late': '#34495e'},  # pink and deep navy blue
    title='Stage',
    s=10, alpha=0.5,
    save='_figure2d_umap_stage.pdf',
)



In [ ]:
# HLA-DRA expression on epithelial lineage
sc.pl.umap(
    epithelial,
    color='HLA-DRA',
    gene_symbols='feature_name',
    title='HLA-DRA',
    use_raw=False,
    cmap='inferno',
    s=2, alpha=0.8,
    save='_figure2_umap_hladra.pdf',
)

## Figure 2a — MHC II expression in malignant vs normal adjacent epithelial cells

Paired comparison of MHC II gene expression between primary tumor cancer cells
and matched normal adjacent epithelial cells. Only donors with both origins
represented are included. Wilcoxon signed-rank test on donor-level means.

In [ ]:
# figure 2a — core MHC II gene set
# nonparametric test used on all genes for consistency (distributions are mixed)
plot_genes_paired_luad(
    luad_paired,
    genes=mhc2_genes,
    test_mode='nonparametric',
    figsize_per_gene=(5, 5),
    save_path=fig_out / 'figure2a_normal_tumor.pdf',
)

In [ ]:
# figure 2a extended — accessory and co-stimulatory molecules
# shows that MHC II processing machinery is also expressed in malignant cells
plot_genes_paired_luad(
    luad_paired,
    genes=extended_genes,
    test_mode='nonparametric',
    nrows=2,
    figsize_per_gene=(5, 5),
    save_path=fig_out / 'figure2a_normal_tumor_extended.pdf',
)

## Figure 2b — MHC II accessory and co-stimulatory molecule expression in malignant cells

Compares expression of MHC II processing and co-stimulatory molecules between
primary tumor cancer cells and normal adjacent epithelial cells. Builds on
prior literature showing these molecules are required for AT2 cell antigen
presentation — their retention in malignant cells suggests preserved antigen
presentation capacity in a subset of tumors.

In [ ]:
# figure 2b — accessory molecules: ICAM1, HLA-DMA, HLA-DMB, CD80, CD86
# subset of extended_genes excluding core MHC II chains
accessory_genes = ['ICAM1', 'HLA-DMA', 'HLA-DMB','CD80', 'CD86']

plot_genes_paired_luad(
    luad_paired,
    genes=accessory_genes,
    test_mode='nonparametric',
    figsize_per_gene=(5, 5),
    save_path=fig_out / 'figure2b_accessory_molecules.pdf',
)

## Figure 2c — MHC II expression in primary tumor vs metastatic cancer cells

Compares sample-level mean MHC II expression between primary tumor and metastatic
cancer cells. Only donors with cells in both origins are included. Mann-Whitney U
test on sample-level means. A UMAP showing the cells being compared is saved
alongside the expression panels.

In [ ]:
# figure 2c UMAP — shows which cells are being compared
sc.pl.umap(
    met_nonmet,
    color='origin',
    palette={'Primary Tumor': 'goldenrod', 'Metastasis': 'red'},
    title='Primary vs metastasis',
    s=3, alpha=0.7,
    show=False,
)
plt.savefig(fig_out / 'figure2c_umap_met_vs_primary.pdf', bbox_inches='tight')
plt.savefig(fig_out / 'figure2c_umap_met_vs_primary.png', bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
# figure 2c — MHC II expression primary tumor vs metastasis
plot_scrna_group_comparison(
    met_nonmet,
    genes=ens_mhc2,
    group_col='origin',
    order=['Primary Tumor', 'Metastasis'],
    palette={'Primary Tumor': 'goldenrod', 'Metastasis': 'red'},
    xtick_labels=['Primary', 'Metastasis'],
    figsize_per_gene=(5, 5),
    nrows=1,
    save_path=fig_out / 'figure2c_met_vs_primary.pdf',
)

## Figure 2d — MHC II expression by disease stage in primary tumor cancer cells

Compares sample-level mean MHC II expression between early (stage I/II) and
late (stage III/IV) primary tumor cancer cells. Mann-Whitney U test on
sample-level means. UICC stage is mapped to early/late before aggregation.

In [ ]:
# figure 2d UMAP — shows which cells are being compared
sc.pl.umap(
    nonmet,
    color='stage_group',
    palette={'Early': '#f8a5c2', 'Late': '#34495e'},
    title='Stage',
    s=3, alpha=0.7,
    show=False,
)
plt.savefig(fig_out / 'figure2d_umap_stage.pdf', bbox_inches='tight')
plt.savefig(fig_out / 'figure2d_umap_stage.png', bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
# figure 2d — MHC II expression early vs late stage primary tumor
plot_scrna_group_comparison(
    nonmet,
    genes=ens_mhc2,
    group_col='stage_group',
    order=['Early', 'Late'],
    palette={'Early': '#f8a5c2', 'Late': '#34495e'},
    xtick_labels=['Early', 'Late'],
    figsize_per_gene=(5, 5),
    nrows=1,
    save_path=fig_out / 'figure2d_early_vs_late.pdf',
)

In [ ]:
def ciita_expr_cell_level_tests(
    adata,
    detection_thresh=0.0,
    s100p_thresh=0.0,
    restrict_ciita_pos=True,
):
    """
    Test whether CIITA expression magnitude differs between S100P+ and S100P-
    epithelial cells at the single-cell level.

    Complements the within-sample delta analysis by testing expression magnitude
    rather than detection frequency. Among cells that express CIITA, are S100P+
    cells expressing less of it?

    Parameters
    ----------
    adata : AnnData
        Subset to paired LUAD epithelial cells (luad_paired).
    detection_thresh : float
        CIITA expression threshold for defining CIITA+ cells (default 0.0).
    s100p_thresh : float
        S100P expression threshold for defining S100P+ cells (default 0.0).
    restrict_ciita_pos : bool
        If True, restrict to CIITA+ cells before testing (recommended).
        Tests expression magnitude rather than detection frequency.

    Returns
    -------
    pd.DataFrame
        One row per origin with columns: n_S100Ppos, n_S100Pneg,
        median_CIITA_S100Ppos, median_CIITA_S100Pneg, p (Mann-Whitney U).
    """
    epi = adata[adata.obs['ann_coarse'] == 'Epithelial cell'].copy()
    epi.obs['origin'] = epi.obs['origin'].astype(str).str.lower().str.strip()

    ciita_id = epi.var.loc[epi.var['feature_name'] == 'CIITA'].index[0]
    s100p_id = epi.var.loc[epi.var['feature_name'] == 'S100P'].index[0]

    Xc = epi[:, ciita_id].X
    Xs = epi[:, s100p_id].X
    ciita = Xc.toarray().ravel() if sp.issparse(Xc) else np.asarray(Xc).ravel()
    s100p = Xs.toarray().ravel() if sp.issparse(Xs) else np.asarray(Xs).ravel()

    df = pd.DataFrame({
        'origin': epi.obs['origin'].values,
        'CIITA':  ciita,
        'S100P':  s100p,
    })
    df['S100P_pos'] = df['S100P'] > s100p_thresh
    df['CIITA_pos'] = df['CIITA'] > detection_thresh

    if restrict_ciita_pos:
        df = df[df['CIITA_pos']].copy()

    out = []
    for origin in ['normal_adjacent', 'tumor_primary']:
        sub = df[df['origin'] == origin]
        a = sub.loc[ sub['S100P_pos'],  'CIITA']
        b = sub.loc[~sub['S100P_pos'],  'CIITA']
        if len(a) == 0 or len(b) == 0:
            out.append({'origin': origin, 'n_S100Ppos': len(a), 'n_S100Pneg': len(b), 'p': np.nan})
            continue
        _, p = mannwhitneyu(a, b, alternative='two-sided')
        out.append({
            'origin':                origin,
            'n_S100Ppos':            len(a),
            'n_S100Pneg':            len(b),
            'median_CIITA_S100Ppos': float(np.median(a)),
            'median_CIITA_S100Pneg': float(np.median(b)),
            'p':                     p,
        })

    return pd.DataFrame(out)

In [ ]:
def plot_ciita_s100p_paired(wide, figsize=(8, 3.8), dpi=150):
    """
    Two-panel paired dot + delta figure comparing CIITA expression in S100P+
    vs S100P- epithelial cells, separately for normal adjacent and tumor.

    Left panel of each pair: per-sample paired dot plot (S100P- vs S100P+).
    Right panel: within-sample delta (S100P+ minus S100P-) with Wilcoxon test.

    Parameters
    ----------
    wide : pd.DataFrame
        Output of ciita_expr_by_s100p_strata_per_sample.
    figsize : tuple
        Figure dimensions in inches.
    dpi : int
        Figure resolution.

    Returns
    -------
    fig : matplotlib.figure.Figure
    """
    origins = ['normal_adjacent', 'tumor_primary']
    labels  = ['Normal adjacent', 'Tumor']

    COLOR_NEG  = '#C4A882'
    COLOR_POS  = '#7B2D8B'
    ALPHA_LINE = 0.35
    DOT_SIZE   = 4

    fig = plt.figure(figsize=figsize, dpi=dpi)
    gs  = fig.add_gridspec(
        1, 5,
        width_ratios=[1.6, 0.85, 0.25, 1.6, 0.85],
        left=0.08, right=0.97, top=0.85, bottom=0.15,
        wspace=0.08,
    )
    ax_slots = [
        fig.add_subplot(gs[0, 0]),
        fig.add_subplot(gs[0, 1]),
        fig.add_subplot(gs[0, 3]),
        fig.add_subplot(gs[0, 4]),
    ]

    for i, (origin, label) in enumerate(zip(origins, labels)):
        ax_paired = ax_slots[i * 2]
        ax_delta  = ax_slots[i * 2 + 1]

        sub = wide[wide['origin'] == origin].dropna(
            subset=['CIITA_mean_S100Pneg', 'CIITA_mean_S100Ppos', 'delta_pos_minus_neg']
        )
        deltas   = sub['delta_pos_minus_neg'].values
        stat, p  = wilcoxon(deltas)
        s        = sig_label(p)
        med      = np.median(deltas)

        # paired dot plot
        rng    = np.random.default_rng(42 + i)
        jitter = rng.uniform(-0.08, 0.08, len(sub))

        for xn, xp, yn, yp in zip(
            jitter, 1 + jitter,
            sub['CIITA_mean_S100Pneg'],
            sub['CIITA_mean_S100Ppos'],
        ):
            ax_paired.plot([xn, xp], [yn, yp],
                           color='grey', lw=0.6, alpha=ALPHA_LINE, zorder=1)

        ax_paired.scatter(jitter,     sub['CIITA_mean_S100Pneg'],
                          color=COLOR_NEG, s=DOT_SIZE, zorder=3)
        ax_paired.scatter(1 + jitter, sub['CIITA_mean_S100Ppos'],
                          color=COLOR_POS, s=DOT_SIZE, zorder=3)

        ax_paired.set_xticks([0, 1])
        ax_paired.set_xticklabels(['S100P-', 'S100P+'], fontsize=8)
        ax_paired.set_xlim(-0.4, 1.4)
        ax_paired.set_ylabel('Mean CIITA (CIITA+ cells)', fontsize=7.5)
        ax_paired.set_title(label, fontsize=9, fontweight='bold', pad=6)
        ax_paired.spines[['top', 'right']].set_visible(False)
        ax_paired.tick_params(labelsize=7)

        # delta strip plot
        rng2 = np.random.default_rng(99 + i)
        jit2 = rng2.uniform(-0.18, 0.18, len(deltas))
        ymin = min(deltas) - np.ptp(deltas) * 0.08
        ymax = max(deltas) + np.ptp(deltas) * 0.45

        ax_delta.set_ylim(ymin, ymax)
        ax_delta.axhline(0, color='black', lw=0.8, ls='--', zorder=1)
        ax_delta.scatter(jit2, deltas, color=COLOR_POS, s=DOT_SIZE, alpha=0.75, zorder=3)
        ax_delta.plot([-0.25, 0.25], [med, med], color='black', lw=2, zorder=4)

        ax_delta.text(0.5, 0.97, f'{s}  p={p:.2e}',
                      ha='center', va='top', fontsize=6.5,
                      transform=ax_delta.transAxes)
        ax_delta.text(0.5, 0.02, f'n={len(sub)}',
                      ha='center', va='bottom', fontsize=6.5, color='grey',
                      transform=ax_delta.transAxes)

        ax_delta.set_xlim(-0.5, 0.5)
        ax_delta.set_xticks([])
        ax_delta.set_ylabel('')
        ax_delta.spines[['top', 'right', 'bottom', 'left']].set_visible(False)
        ax_delta.tick_params(axis='y', labelsize=7, left=False, labelleft=False)

    fig.suptitle('CIITA expression in S100P+ vs S100P- epithelial cells',
                 fontsize=9, y=0.97)
    retu

In [ ]:
# within-sample delta: mean CIITA in S100P+ vs S100P- cells, among CIITA+ cells only
# tests whether S100P status predicts CIITA expression level within the same sample
wide = ciita_expr_by_s100p_strata_per_sample(luad_paired, agg='mean_detected_only')

for origin in ['normal_adjacent', 'tumor_primary']:
    sub = wide[wide['origin'] == origin].dropna(subset=['delta_pos_minus_neg'])
    if len(sub) == 0:
        print(f'{origin}: no samples with both S100P strata represented')
        continue
    stat, p = wilcoxon(sub['delta_pos_minus_neg'])
    print(
        f'{origin}  n_samples={len(sub)}'
        f'  Wilcoxon(delta) p={p:.3e}'
        f'  median_delta={np.nanmedian(sub["delta_pos_minus_neg"]):.3g}'
    )

In [ ]:
# cell-level test: among CIITA+ cells, is expression magnitude lower in S100P+ cells?
# Mann-Whitney U, run separately per origin
ciita_expr_cell_level_tests(luad_paired, restrict_ciita_pos=True)

In [ ]:
fig = plot_ciita_s100p_paired(wide)
fig.savefig(fig_out / 'ciita_s100p_paired.pdf', bbox_inches='tight')
fig.savefig(fig_out / 'ciita_s100p_paired.png', bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
# figure 2c — MHC II expression in primary tumor vs metastatic cancer cells
# restricted to cancer cells (ann_fine == 'Cancer cells') in both origins
met_nonmet = cancer_non_cancer[
    cancer_non_cancer.obs['origin'].isin(['tumor_primary', 'tumor_metastasis']) &
    (cancer_non_cancer.obs['ann_fine'] == 'Cancer cells')
].copy()

plot_scrna_group_comparison(
    met_nonmet,
    genes=ens_mhc2,
    group_col='origin',
    order=['tumor_primary', 'tumor_metastasis'],
    palette={'tumor_primary': 'goldenrod', 'tumor_metastasis': 'red'},
    xtick_labels=['Primary tumor', 'Metastasis'],
    figsize_per_gene=(5, 5),
    nrows=1,
    save_path=fig_out / 'figure2c_met_vs_primary.pdf',
)

In [ ]:
# figure 2d — MHC II expression by UICC stage in primary tumor cancer cells
# stage is mapped to early (I/II) vs late (III/IV) before aggregation
nonmet = met_nonmet[met_nonmet.obs['origin'] == 'tumor_primary'].copy()

# map UICC stage to early/late — done at cell level before aggregation
nonmet.obs['stage_group'] = (
    nonmet.obs['uicc_stage']
    .map(lambda x: 'Early' if x in ['I', 'II'] else 'Late')
)

plot_scrna_group_comparison(
    nonmet,
    genes=ens_mhc2,
    group_col='stage_group',
    order=['Early', 'Late'],
    palette={'Early': '#f8a5c2', 'Late': '#e55039'},
    xtick_labels=['Early stage', 'Late stage'],
    figsize_per_gene=(5, 5),
    nrows=1,
    save_path=fig_out / 'figure2d_early_vs_late.pdf',
)